# 0. 라이브러리 호출

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from google.cloud import bigquery

In [2]:
PROJECT_ID = "sns-analysis-prj"
DATA_SET = "sns_analysis"

client = bigquery.Client(project=PROJECT_ID)

## questionreport (질문 신고(평가))

In [3]:
# 전처리 수행 대상 테이블 호출
sql = f"""
    SELECT *
    FROM `{PROJECT_ID}.{DATA_SET}.polls_questionreport`
"""

# 판다스 데이터프레임으로 변환
df = client.query(sql).to_dataframe()

print(df.head())

      id reason                created_at  question_id  user_id
0   4852  그냥 싫어 2023-05-07 12:14:13+00:00           99   894226
1   4971  그냥 싫어 2023-05-07 13:43:20+00:00           99   906185
2   5389  그냥 싫어 2023-05-08 02:16:54+00:00           99   944035
3   7884  그냥 싫어 2023-05-09 14:19:10+00:00           99   981801
4  11094  그냥 싫어 2023-05-11 13:26:01+00:00           99   887923


In [4]:
df.isna().sum()

id             0
reason         0
created_at     0
question_id    0
user_id        0
dtype: int64

In [5]:
df[['reason', 'created_at', 'question_id', 'user_id']].duplicated().sum()

np.int64(0)

In [6]:
df.value_counts('reason')

reason
그냥 싫어                   28446
나랑 맞지 않는 질문인 것 같음        9541
불쾌한 질문 내용                5386
자꾸 같은 내용의 질문 반복          3202
어떻게 이런 생각을? 이 질문 최고!     1821
한 친구가 질문을 반복적으로 보냄       1701
기타                        480
이 질문은 재미없어요               471
불쾌한 내용이 포함되어 있음           250
오타가 있음                     68
선정적이거나 자극적인 질문             58
Name: count, dtype: int64

* 결측이나 중복값은 없는 것으로 파악됩니다.
* 따라서 해당 단계에서는 별도의 추가 전처리를 수행하지 않습니다.

## question (질문)

In [7]:
# 전처리 수행 대상 테이블 호출
sql2 = f"""
    SELECT *
    FROM `{PROJECT_ID}.{DATA_SET}.polls_question`
"""

# 판다스 데이터프레임으로 변환
df2 = client.query(sql2).to_dataframe()

print(df2.head())

    id                 question_text                created_at
0   99            가장 신비한 매력이 있는 사람은? 2023-03-31 15:22:53+00:00
1  100  "이 사람으로 한 번 살아보고 싶다" 하는 사람은? 2023-03-31 15:22:53+00:00
2  101                     미래의 틱톡커는? 2023-03-31 15:22:54+00:00
3  102               여기서 제일 특이한 친구는? 2023-03-31 15:22:54+00:00
4  103               가장 지켜주고 싶은 사람은? 2023-03-31 15:22:55+00:00


In [8]:
df2.isna().sum()

id               0
question_text    0
created_at       0
dtype: int64

In [10]:
df2[df2[['question_text', 'created_at']].duplicated()]

,id,question_text,created_at
1543,1652,vote,2023-06-02 08:06:23+00:00
1544,1653,vote,2023-06-02 08:06:23+00:00
1545,1654,vote,2023-06-02 08:06:23+00:00
1546,1655,vote,2023-06-02 08:06:23+00:00
1547,1656,vote,2023-06-02 08:06:23+00:00
1548,1657,vote,2023-06-02 08:06:23+00:00
1549,1658,vote,2023-06-02 08:06:23+00:00
1561,1670,회사 비서에 잘 어울리는 사람,2023-06-02 08:06:23+00:00
1640,1749,vote,2023-06-02 08:06:26+00:00
1664,1773,플래너 잘 쓸 것 같은 사람은?,2023-06-02 08:06:26+00:00


* 'vote'라는 질문을 포함해서 중복으로 질문이 쌓여있는 질문셋을 확인하였습니다.

In [12]:
df2[
    df2[['question_text', 'created_at']].duplicated(keep=False)
    & (df2['question_text'] != 'vote')
]

,id,question_text,created_at
1560,1669,회사 비서에 잘 어울리는 사람,2023-06-02 08:06:23+00:00
1561,1670,회사 비서에 잘 어울리는 사람,2023-06-02 08:06:23+00:00
1663,1772,플래너 잘 쓸 것 같은 사람은?,2023-06-02 08:06:26+00:00
1664,1773,플래너 잘 쓸 것 같은 사람은?,2023-06-02 08:06:26+00:00
1820,1929,집안일을 가장 잘 할 것 같은 사람은?,2023-06-02 08:06:32+00:00
1821,1930,집안일을 가장 잘 할 것 같은 사람은?,2023-06-02 08:06:32+00:00
1897,2006,제일 마음씨 착할 것 같은 친구,2023-06-02 08:06:34+00:00
1898,2007,제일 마음씨 착할 것 같은 친구,2023-06-02 08:06:34+00:00
1942,2051,잠버릇이 특이할 것 같은 친구,2023-06-02 08:06:35+00:00
1943,2052,잠버릇이 특이할 것 같은 친구,2023-06-02 08:06:35+00:00


In [13]:
df2[
    df2[['question_text', 'created_at']].duplicated(keep=False)
    & (df2['question_text'] == 'vote')
]

,id,question_text,created_at
1542,1651,vote,2023-06-02 08:06:23+00:00
1543,1652,vote,2023-06-02 08:06:23+00:00
1544,1653,vote,2023-06-02 08:06:23+00:00
1545,1654,vote,2023-06-02 08:06:23+00:00
1546,1655,vote,2023-06-02 08:06:23+00:00
1547,1656,vote,2023-06-02 08:06:23+00:00
1548,1657,vote,2023-06-02 08:06:23+00:00
1549,1658,vote,2023-06-02 08:06:23+00:00
1638,1747,vote,2023-06-02 08:06:26+00:00
1640,1749,vote,2023-06-02 08:06:26+00:00


In [15]:
question_report_counts = (
    df.groupby('question_id')['id']
    .nunique()
    .reset_index(name='신고 횟수')
    .sort_values('신고 횟수', ascending=False)
)

display(question_report_counts)

,question_id,신고 횟수
12,111,988
311,410,803
299,398,660
325,424,533
206,305,493
...,...,...
2439,2980,1
1801,2019,1
2441,2983,1
2443,2986,1


In [36]:
# 중복 질문이거나 question_text가 vote인 질문
target_question_mask = (
    df2[['question_text', 'created_at']].duplicated(keep=False)
    | df2['question_text'].eq('vote')
)

target_questions = (
    df2.loc[
        target_question_mask,
        ['id', 'question_text', 'created_at'],
    ]
    .copy()
)

In [37]:
# 질문별 고유 신고 횟수
question_report_counts = (
    df.groupby('question_id')['id']
    .nunique()
    .reset_index(name='신고 횟수')
)

target_questions_with_reports = (
    target_questions
    .merge(
        question_report_counts,
        left_on='id',
        right_on='question_id',
        how='left',
        validate='one_to_one',
    )
    .drop(columns='question_id')
    .rename(columns={'id': 'question_id'})
)

target_questions_with_reports['신고 횟수'] = (
    target_questions_with_reports['신고 횟수']
    .fillna(0)
    .astype(int)
)

In [38]:
target_questions_with_reports['질문 구분'] = np.where(
    target_questions_with_reports['question_text'].eq('vote'),
    'vote',
    '중복 질문',
)

target_questions_with_reports = (
    target_questions_with_reports
    .sort_values(
        ['질문 구분', '신고 횟수'],
        ascending=[True, False],
    )
    .reset_index(drop=True)
)

with pd.option_context(
    'display.max_rows', None,
    'display.max_columns', None,
    'display.max_colwidth', None,
):
    display(target_questions_with_reports)

,question_id,question_text,created_at,신고 횟수,질문 구분
0,483,vote,2023-05-02 05:33:11+00:00,73,vote
1,186,vote,2023-04-01 11:09:15+00:00,45,vote
2,881,vote,2023-05-15 13:59:44+00:00,39,vote
3,1232,vote,2023-05-15 14:02:08+00:00,36,vote
4,696,vote,2023-05-15 13:58:24+00:00,34,vote
5,807,vote,2023-05-15 13:59:11+00:00,34,vote
6,736,vote,2023-05-15 13:58:40+00:00,32,vote
7,1139,vote,2023-05-15 14:01:32+00:00,32,vote
8,1320,vote,2023-05-15 14:02:46+00:00,21,vote
9,940,vote,2023-05-15 14:00:10+00:00,20,vote


* polls_question 데이터의 중복값 및 vote 데이터에 대한 전처리 여부를 검토했습니다.

* 최초에는 중복 질문과 vote 데이터를 제외하려고 했으나, 데이터를 추가로 확인한 결과 동일한 내용의 질문이 서로 다른 id로 존재하며, 각각에 신고 기록이 발생한 사례가 확인되었습니다.
* 이를 고려하면 중복으로 보이는 질문들도 실제로 각각 사용자에게 노출되었을 가능성이 있으며, vote 데이터 역시 현재 정보만으로 실제 노출 여부를 명확하게 구분하기 어렵다고 판단했습니다.
* 따라서 해당 데이터를 임의로 제거할 경우 질문 노출, 신고 등 다른 분석 결과에 영향을 줄 가능성이 있어, 별도의 삭제 전처리는 진행하지 않기로 했습니다.
* 중복 질문 및 vote 데이터가 존재한다는 데이터 특이사항만 공유드리며, 이후 분석 목적에 따라 필요한 경우 별도로 처리하는 방향이 적절할 것 같습니다.